In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder.appName("CustomerOrderInsightsPySpark").getOrCreate()

In [2]:
# Creating CSV Datasets
orders_csv_data = """order_id,customer_id,delivery_date,delivery_issue,delay_days,is_delayed
501,1001,2026-06-14,Unreported Issue,12,1
502,1002,2026-06-25,Sorting Hub Delay,1,1
503,1003,2026-06-16,Unreported Issue,10,1
504,1004,2026-06-26,Damaged in Transit,0,0
506,1001,2026-06-20,Unreported Issue,6,1
507,1002,2026-06-27,Sorting Hub Delay,-1,0
509,1004,2026-06-10,Unreported Issue,16,1
510,1005,2026-06-24,Unreported Issue,2,1
511,1001,2026-06-15,Sorting Hub Delay,11,1
512,1002,2026-06-05,Unreported Issue,21,1
513,1003,2026-06-26,Incorrect Address,0,0
514,1004,2026-06-18,Unreported Issue,8,1
515,1005,2026-06-25,Damaged in Transit,1,1
"""
with open("orders.csv", "w") as f:
    f.write(orders_csv_data)

customers_csv_data = """customer_id,customer_name,delivery_city,customer_tier
1001,Alice Smith,New York,Gold
1002,Bob Jones,Los Angeles,Silver
1003,Charlie Brown,Chicago,Bronze
1004,Diana Prince,Houston,Gold
1005,Evan Wright,Miami,Silver
"""
with open("customers.csv", "w") as f:
    f.write(customers_csv_data)

In [4]:
# Loading Data
orders_df = spark.read.csv("orders.csv", header=True, inferSchema=True)
orders_df.show()

customers_df = spark.read.csv("customers.csv", header=True, inferSchema=True)
customers_df.show()

print("Schema of orders:")
orders_df.printSchema()

print("Schema of customers:")
customers_df.printSchema()

+--------+-----------+-------------+------------------+----------+----------+
|order_id|customer_id|delivery_date|    delivery_issue|delay_days|is_delayed|
+--------+-----------+-------------+------------------+----------+----------+
|     501|       1001|   2026-06-14|  Unreported Issue|        12|         1|
|     502|       1002|   2026-06-25| Sorting Hub Delay|         1|         1|
|     503|       1003|   2026-06-16|  Unreported Issue|        10|         1|
|     504|       1004|   2026-06-26|Damaged in Transit|         0|         0|
|     506|       1001|   2026-06-20|  Unreported Issue|         6|         1|
|     507|       1002|   2026-06-27| Sorting Hub Delay|        -1|         0|
|     509|       1004|   2026-06-10|  Unreported Issue|        16|         1|
|     510|       1005|   2026-06-24|  Unreported Issue|         2|         1|
|     511|       1001|   2026-06-15| Sorting Hub Delay|        11|         1|
|     512|       1002|   2026-06-05|  Unreported Issue|        2

In [5]:
# Joining orders and customer tables
orders_customers_df = orders_df.join(customers_df, "customer_id", "inner")
orders_customers_df.show()

+-----------+--------+-------------+------------------+----------+----------+-------------+-------------+-------------+
|customer_id|order_id|delivery_date|    delivery_issue|delay_days|is_delayed|customer_name|delivery_city|customer_tier|
+-----------+--------+-------------+------------------+----------+----------+-------------+-------------+-------------+
|       1001|     501|   2026-06-14|  Unreported Issue|        12|         1|  Alice Smith|     New York|         Gold|
|       1002|     502|   2026-06-25| Sorting Hub Delay|         1|         1|    Bob Jones|  Los Angeles|       Silver|
|       1003|     503|   2026-06-16|  Unreported Issue|        10|         1|Charlie Brown|      Chicago|       Bronze|
|       1004|     504|   2026-06-26|Damaged in Transit|         0|         0| Diana Prince|      Houston|         Gold|
|       1001|     506|   2026-06-20|  Unreported Issue|         6|         1|  Alice Smith|     New York|         Gold|
|       1002|     507|   2026-06-27| Sor

In [6]:
# Group by region to count delays
print("Grouping by City To Count Delays:")
df_city_delays = orders_customers_df.filter(col("is_delayed") == 1) \
  .groupBy("delivery_city").agg(sum("is_delayed").alias("delayed_orders_count")) \
  .orderBy(col("delayed_orders_count").desc())
df_city_delays.show()

Grouping by City To Count Delays:
+-------------+--------------------+
|delivery_city|delayed_orders_count|
+-------------+--------------------+
|     New York|                   3|
|  Los Angeles|                   2|
|      Houston|                   2|
|        Miami|                   2|
|      Chicago|                   1|
+-------------+--------------------+



In [8]:
# Save results to a file
df_city_delays.write.mode("overwrite").csv("delayed_orders_by_city", header=True)
print("Results saved successfully!")

Results saved successfully!
